In [4]:
from pathlib import Path
from time import perf_counter
from typing import Callable

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import RYGate, StatePreparation
from qiskit_aer import AerSimulator

import geqie
from geqie.encodings import frqi

pio.templates["latex"] = pio.templates["plotly_white"].update(layout=dict(font=dict(family="CMU Serif", size=14)))
pio.templates.default = "latex"


In [6]:
SIDES = [2, 4, 8, 16, 32, 64]
SEED = 0
BASIS_GATES = ["cx", "u"]
OPTIMIZATION_LEVEL = 1
RESULTS_PATH = Path(".").parent / "results"
CSV_PATH = RESULTS_PATH / "circuit_cost_frqi.csv"
PLOT_PATH = RESULTS_PATH / "plots" / "circuit_cost_frqi.pdf"


In [7]:
def geqie_circuit(image: np.ndarray) -> QuantumCircuit:
    return geqie.encode(frqi.init_function, frqi.data_function, frqi.map_function, image, perform_measurement=False)


def gate_level_circuit(image: np.ndarray) -> QuantumCircuit:
    """Original FRQI construction: Hadamards on the position register and one multi-controlled R_y per pixel."""
    n_position = 2 * int(np.log2(image.shape[0]))
    position = list(range(1, n_position + 1))
    circuit = QuantumCircuit(n_position + 1)
    circuit.h(position)
    for m, angle in enumerate(frqi_angles(image)):
        zero_bits = [qubit for k, qubit in enumerate(position) if not (m >> k) & 1]
        if zero_bits:
            circuit.x(zero_bits)
        circuit.append(RYGate(2 * angle).control(n_position), [*position, 0])
        if zero_bits:
            circuit.x(zero_bits)
    return circuit


def state_preparation_circuit(image: np.ndarray) -> QuantumCircuit:
    state = frqi_state(image)
    circuit = QuantumCircuit(int(np.log2(state.size)))
    circuit.append(StatePreparation(state), circuit.qubits)
    return circuit



In [8]:
def frqi_angles(image: np.ndarray) -> np.ndarray:
    return image.flatten() / 255.0 * (np.pi / 2)


def frqi_state(image: np.ndarray) -> np.ndarray:
    """FRQI state with the value qubit as the least significant qubit, matching the GEQIE qubit order."""
    angles = frqi_angles(image)
    return np.stack([np.cos(angles), np.sin(angles)], axis=1).flatten() / np.sqrt(angles.size)


def probabilities(circuit: QuantumCircuit) -> np.ndarray:
    circuit = circuit.copy()
    circuit.save_statevector()
    statevector = AerSimulator(method="statevector").run(circuit).result().get_statevector()
    return np.abs(np.asarray(statevector)) ** 2


def classical_fidelity(p: np.ndarray, q: np.ndarray) -> float:
    return float(np.sum(np.sqrt(p * q)) ** 2)


In [9]:
def run_benchmark() -> pd.DataFrame:
    methods: dict[str, Callable[[np.ndarray], QuantumCircuit]] = {
        "GEQIE": geqie_circuit,
        "Gate-level reference": gate_level_circuit,
        "Qiskit StatePreparation": state_preparation_circuit,
    }
    rng = np.random.default_rng(SEED)
    rows = []
    for side in SIDES:
        image = rng.integers(0, 256, size=(side, side), dtype=np.uint8)
        expected = np.abs(frqi_state(image)) ** 2
        for method, build in methods.items():
            start = perf_counter()
            circuit = build(image)
            built = perf_counter()
            transpiled = transpile(circuit, basis_gates=BASIS_GATES, optimization_level=OPTIMIZATION_LEVEL)
            transpiled_at = perf_counter()
            rows.append({
                "method": method,
                "side": side,
                "pixels": side**2,
                "n_qubits": transpiled.num_qubits,
                "cnot_count": transpiled.count_ops().get("cx", 0),
                "depth": transpiled.depth(),
                "build_time_s": built - start,
                "transpile_time_s": transpiled_at - built,
                "total_time_s": transpiled_at - start,
                "fidelity": classical_fidelity(probabilities(transpiled), expected),
            })
            print(rows[-1])
    return pd.DataFrame(rows)


In [10]:
def plot(results: pd.DataFrame) -> None:
    results = results.assign(size=[f"{side}×{side}<br>({n} qubits)" for side, n in zip(results["side"], results["n_qubits"])])
    long = results.melt(
        id_vars=["method", "size"],
        value_vars=["cnot_count", "total_time_s"],
        var_name="metric",
        value_name="value",
    )
    titles = {"cnot_count": "(a) CNOT count", "total_time_s": "(b) Generation time [s]"}
    fig = px.line(
        long,
        x="size",
        y="value",
        color="method",
        symbol="method",
        facet_col="metric",
        markers=True,
        log_y=True,
        color_discrete_sequence=px.colors.qualitative.D3,
        symbol_sequence=["circle", "diamond", "square"],
        labels={"size": "Image size", "method": ""},
    )
    fig.for_each_annotation(lambda annotation: annotation.update(text=titles[annotation.text.split("=")[-1]]))
    fig.update_yaxes(matches=None, showticklabels=True, title=None, exponentformat="power", dtick=1)
    fig.update_xaxes(tickfont=dict(size=12))
    fig.update_layout(
        width=900,
        height=400,
        margin=dict(l=5, r=5, t=30, b=5),
        legend=dict(orientation="h", x=0.5, xanchor="center", y=-0.3),
    )
    PLOT_PATH.parent.mkdir(parents=True, exist_ok=True)
    fig.write_image(PLOT_PATH)
    fig.show()


In [12]:
run_benchmark().to_csv(CSV_PATH, index=False)
plot(pd.read_csv(CSV_PATH))


{'method': 'GEQIE', 'side': 2, 'pixels': 4, 'n_qubits': 3, 'cnot_count': 6, 'depth': 12, 'build_time_s': 0.001802199985831976, 'transpile_time_s': 0.009327400010079145, 'total_time_s': 0.011129599995911121, 'fidelity': 1.0}
{'method': 'Gate-level reference', 'side': 2, 'pixels': 4, 'n_qubits': 3, 'cnot_count': 32, 'depth': 49, 'build_time_s': 0.0011372000444680452, 'transpile_time_s': 0.005796400015242398, 'total_time_s': 0.006933600059710443, 'fidelity': 1.0000000000000004}
{'method': 'Qiskit StatePreparation', 'side': 2, 'pixels': 4, 'n_qubits': 3, 'cnot_count': 3, 'depth': 7, 'build_time_s': 0.00017090002074837685, 'transpile_time_s': 0.00880369998048991, 'total_time_s': 0.008974600001238286, 'fidelity': 1.0000000000000009}
{'method': 'GEQIE', 'side': 4, 'pixels': 16, 'n_qubits': 5, 'cnot_count': 64, 'depth': 106, 'build_time_s': 0.0017194999381899834, 'transpile_time_s': 0.009677499998360872, 'total_time_s': 0.011396999936550856, 'fidelity': 1.0000000000000027}
{'method': 'Gate-lev